In [ ]:
import pandas as pd
import numpy as np
import yaml
from sklearn.preprocessing import quantile_transform
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def trait_INT_zscore(trait, phenotype_data_dir):
    pheno_dt = pd.read_parquet(
        f"{phenotype_data_dir}/{trait}/data.parquet"
    )
    trait_mean = pheno_dt[f"{trait}_raw"].mean()
    trait_sd = pheno_dt[f"{trait}_raw"].std()
    pheno_dt["zscore"] = (pheno_dt[f"{trait}_raw"] - trait_mean) / trait_sd
    pheno_dt["raw"] = pheno_dt[f"{trait}_raw"]
    pheno_dt["INT"] = pheno_dt[f"{trait}"]
    pheno_dt["new_INT"] = quantile_transform(
        pheno_dt[["zscore"]], output_distribution="normal", random_state=0, copy=True
    )
    pheno_dt["trait"] = trait

    pheno_dt = pheno_dt.rename(columns={"eid": "individual"})
    # pheno_dt["individual"] = pheno_dt["individual"].astype("str")
    # pheno_dt = pheno_dt.set_index('individual')
    return pheno_dt[["individual", "trait", "raw", "zscore", "INT", "new_INT"]]

In [ ]:
config_path = '../run_config_local.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

TRAITS=config.get('traits')

phenotype_dir = '/s/project/uk_biobank/clean/decoded_phenotypes'


In [ ]:
trait_zscore_list = []
trait_int_list = []
for trait in TRAITS:
    raw_pheno = trait_INT_zscore(trait, phenotype_dir).set_index('individual')

     # Check for duplicates in the index
    if raw_pheno.index.has_duplicates:
        print(f"Warning: Duplicate individual IDs found for trait '{trait}'.")
        raw_pheno = raw_pheno[~raw_pheno.index.duplicated(keep='first')]

    raw_pheno[trait] = raw_pheno['zscore']
    trait_zscore_list.append(raw_pheno[[trait]])
    raw_pheno[trait] = raw_pheno['new_INT']
    trait_int_list.append(raw_pheno[[trait]])

traits_zscore_df = pd.concat(trait_zscore_list, axis=1)
traits_int_df = pd.concat(trait_int_list, axis=1)

In [ ]:
# Compute the correlation matrix for the INT traits
correlation_matrix_int = traits_int_df.corr()

# Display the correlation matrix (optional)
print("Correlation matrix for INT traits:")

# Create the heatmap
# plt.figure(figsize=(15, 12)) # Adjust size as needed
# sns.heatmap(correlation_matrix_int, cmap='coolwarm', center=0, annot=False) # annot=True might be too crowded for 41 traits
sns.clustermap(correlation_matrix_int,
               center=0,
               cmap='coolwarm',
               figsize=(13, 13),
               xticklabels=True,
               yticklabels=True)
# plt.title('Correlation Matrix for Traits (INT)')
plt.show()


In [ ]:
corr_mat_melt = correlation_matrix_int.reset_index(names=['trait1']).melt(id_vars=['trait1'], var_name='trait2', value_name='correlation')
# Filter out self-correlations (trait1 == trait2) and symmetric duplicates (keep trait1 < trait2)
unique_corr_pairs = corr_mat_melt[corr_mat_melt['trait1'] < corr_mat_melt['trait2']]
high_corr_pairs = unique_corr_pairs[(abs(unique_corr_pairs.correlation) > 0.5)]
high_corr_pairs

high_corr_traits = set(high_corr_pairs.trait1).union(set(high_corr_pairs.trait2))

r2 = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/old_results/v1NEWsplit_deepRVAT_testsplit0.25_r2.pq')
r2 = r2[r2['model'] == 'funcrvp_omics_pops']
r2.head()

In [ ]:
corr_mat_melt.query('correlation > 0.9')

In [ ]:
def choose_best_trait(row, r2):
    trait1 = row['trait1']
    trait2 = row['trait2']
    r2_trait1 = r2[r2['trait'] == trait1]['rel_delta_r2'].values
    r2_trait2 = r2[r2['trait'] == trait2]['rel_delta_r2'].values
    if len(r2_trait1) > 0 and len(r2_trait2) > 0:
        r2_trait1_value = r2_trait1[0]
        r2_trait2_value = r2_trait2[0]

        if r2_trait1_value >= r2_trait2_value:
            return trait2
        else:
            return trait1
    elif len(r2_trait1) > 0:
        return trait2
    elif len(r2_trait2) > 0:
        return trait1
    else:
        return None

drop_list = high_corr_pairs.apply(choose_best_trait, axis=1, r2=r2)
drop_list
# drop_list_2 = drop_list_2.dropna().tolist()

# drop_list = ['Apolipoprotein_B', 'Creatinine', 'Apolipoprotein_A', 'Cholesterol', 'Reticulocyte_count', 'Total_bilirubin', 'Platelet_count', 'Mean_corpuscular_volume']

unique_corr_pairs[(~unique_corr_pairs.trait1.isin(drop_list)) & (~unique_corr_pairs.trait2.isin(drop_list)) & (abs(unique_corr_pairs.correlation) > 0.5)]

set(drop_list)


In [ ]:
unique_corr_pairs